# NULL Handling

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que1: Unfinished Parts in Assembly

**Difficulty:** Easy

### Problem

A manufacturing plant tracks every part moving through its assembly line. Each part records the date its assembly started and, once complete, the date it finished. A part is still unfinished when it has a start date but no recorded finish date. List each unfinished part's name (`part_name`) and the assembly stage it stopped at (`assembly_step`), ordered by `part_id`.

**Schema columns:** `parts_assembly.part_id`, `parts_assembly.part_name`, `parts_assembly.assembly_step`, `parts_assembly.start_date`, `parts_assembly.finish_date`

**Output columns:** `part_name`, `assembly_step`

### Examples

#### Example 1

**Input:**

**parts_assembly:**

| part_id | part_name | assembly_step | start_date | finish_date |
|--------:|-----------|-------------:|------------|-------------|
| 1 | Transmission | 3 | 2024-01-01 | NULL |
| 2 | Engine Block | 5 | 2023-12-15 | 2024-01-15 |
| 3 | Suspension | 2 | 2024-01-05 | NULL |
| 4 | Chassis | 4 | 2024-01-03 | 2024-01-10 |
| 5 | Turbo | 1 | 2024-01-07 | NULL |

**Output:**

| part_name | assembly_step |
|-----------|-------------:|
| Transmission | 3 |
| Suspension | 2 |
| Turbo | 1 |

**Explanation:** Transmission, Suspension, and Turbo each have a start date but no finish date, so they are still unfinished and appear in the result. Engine Block and Chassis both have finish dates, so they are excluded. Rows are returned ordered by `part_id`: 1, 3, 5.

### Constraints

- A part is unfinished when `finish_date` has no value.
- Sort the results by `part_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
parts_assembly_data = [(1,"Transmission",3,"2024-01-01",None),(2,"Engine Block",5,"2023-12-15","2024-01-15"),(3,"Suspension",2,"2024-01-05",None),(4,"Chassis",4,"2024-01-03","2024-01-10"),(5,"Turbo",1,"2024-01-07",None)]
parts_assembly_df = spark.createDataFrame(parts_assembly_data, ["part_id","part_name","assembly_step","start_date","finish_date"])

display(parts_assembly_df)

output_df = parts_assembly_df.filter(col("finish_date").isNull()).orderBy("part_id")

display(output_df)


## Que2: Inventory Fill Forward

**Difficulty:** Easy

### Problem

A warehouse records stock levels periodically, leaving `stock_level` null when no new count is taken. For each product, replace a missing stock level with its most recent non-null level. Return all rows ordered by `product_id`, then `date`.

**Schema columns:** `inventory_levels.date`, `inventory_levels.product_id`, `inventory_levels.stock_level`

**Output columns:** `date`, `product_id`, `stock_level_filled`

### Examples

#### Example 1

**Input:**

**inventory_levels:**

| date | product_id | stock_level |
|------------|-----------|------------:|
| 2024-01-01 | A | 100 |
| 2024-01-02 | A | NULL |
| 2024-01-03 | A | 110 |
| 2024-01-04 | A | NULL |
| 2024-01-05 | A | 120 |

**Output:**

| date | product_id | stock_level_filled |
|------------|-----------|-------------------:|
| 2024-01-01 | A | 100.0 |
| 2024-01-02 | A | 100.0 |
| 2024-01-03 | A | 110.0 |
| 2024-01-04 | A | 110.0 |
| 2024-01-05 | A | 120.0 |

**Explanation:** January 2 and January 4 inherit product A's most recent recorded stock levels.

### Constraints

- Fill values independently for each product.
- A leading null remains null when no earlier stock level exists.
- Return results matching the expected output schema and order.

In [0]:
inventory_levels_data = [("2024-01-01", "A", None),("2024-01-02", "A", None),("2024-01-03", "A", 100),("2024-01-04", "A", None),("2024-01-05", "A", None),("2024-01-06", "A", 110),("2024-01-07", "A", None),("2024-01-08", "A", 90),("2024-01-09", "A", None),("2024-01-10", "A", None),("2024-01-01", "B", None),("2024-01-02", "B", 500),("2024-01-03", "B", None),("2024-01-04", "B", None),("2024-01-01", "C", None),("2024-01-02", "C", None),("2024-01-03", "C", None),("2024-01-01", "D", 50),("2024-01-02", "D", 60),("2024-01-03", "D", 70),("2024-01-01", "E", 1000),("2024-01-02", "E", None),("2024-01-03", "E", None),("2024-01-04", "E", None),("2024-01-05", "E", 800),("2024-01-03", "F", 300),("2024-01-01", "F", 100),("2024-01-04", "F", None),("2024-01-02", "F", None),("2024-01-05", "F", 500),("2024-01-01", "G", 10),("2024-01-02", "G", None),("2024-01-03", "G", 20),("2024-01-04", "G", None),("2024-01-05", "G", 30),("2024-01-06", "G", None)]

inventory_levels_df = spark.createDataFrame(inventory_levels_data,["date", "product_id", "stock_level"])

window = Window.partitionBy("product_id").orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = (
inventory_levels_df
    .withColumn("stock_last_value", last("stock_level", True).over(window))
)

display(df)

## Que3: Second Highest Salary

**Difficulty:** Medium

### Problem

Write a solution to retrieve the second highest unique salary from the `Employee` table. If there is no second highest salary, return null.

**Schema columns:** `fss_employee.id`, `fss_employee.salary`

**Output columns:** `SecondHighestSalary`

### Examples

#### Example 1

**Input:**

**fss_employee:**

| id | salary |
|---:|-------:|
| 1 | 100 |
| 2 | 200 |
| 3 | 300 |

**Output:**

| SecondHighestSalary |
|--------------------:|
| 200 |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
fss_employee_data = [(1,100),(2,200),(3,300)]
fss_employee_df = spark.createDataFrame(fss_employee_data, ["id","salary"])

display(fss_employee_df)

window = Window.orderBy(col("salary").desc())

output_df = fss_employee_df.withColumn("rnk", dense_rank().over(window)).filter(col("rnk") == 2).select(col("salary").alias("second_highest_salary"))

display(output_df)
